In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s

anchors = s._load_anchors()
questions = ["أوصل للجولد إزاي؟", "عايز أحجز وأستخدم نقطي كخصم", "ينفع أشرب شيشة في الأوضة؟"]

for text in questions:
    q = s._encode(text)
    a = s._anchor_vecs @ q
    c = s._vectors @ q
    ai = int(a.argmax())
    ci = int(c.argmax())
    print(text)
    print("   anchor", round(float(a[ai]), 3), "|", anchors[ai])
    print("   chunk ", round(float(c[ci]), 3), "|", s._chunks[ci]["source"], "|", s._chunks[ci]["text"][:60].replace("\n", " "))
    print("   diff  ", round(float(a[ai] - c[ci]), 3))

c:\projects\HotelAI-Intelligent-Agent_Final_Data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 22:47:59.198000 25852 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 30119.98it/s]


أوصل للجولد إزاي؟
   anchor 0.388 | حاسس بوجع في صدري، آخد إيه؟
   chunk  0.296 | contact_information.txt | Q: أكلم الحجوزات على رقم إيه؟ A: عنوان الفندق 14 شارع كورنيش
   diff   0.091
عايز أحجز وأستخدم نقطي كخصم
   anchor 0.579 | تقدر تحجزلي طيران للقاهرة كمان؟
   chunk  0.523 | booking_process.txt | Q: عايز أحجز، أعمل إيه؟ A: تختار التواريخ ونوع الأوضة، تدّين
   diff   0.055
ينفع أشرب شيشة في الأوضة؟
   anchor 0.582 | I have chest pain, what should I take for it?
   chunk  0.525 | complaints_resolution.txt | Q: دخلت الأوضة لقيتها مش نضيفة خالص ومش هقدر أقعد فيها A: كل
   diff   0.057


In [2]:
from my_rag.evaluate import load_tests

positives, refuse, faq_out = load_tests()

def diff(text):
    q = s._encode(text)
    return float((s._anchor_vecs @ q).max() - (s._vectors @ q).max())

inside = []
for item in positives:
    inside.append((round(diff(item["question"]), 3), item["question"]))

outside = []
for t in refuse + faq_out:
    if not isinstance(t, str):
        t = t["question"]
    outside.append((round(diff(t), 3), t))

inside.sort(reverse=True)
outside.sort()

print("in-scope with the HIGHEST diff (closest to being blocked):")
for d, t in inside[:8]:
    print("   ", d, t)

print("\noutside with the LOWEST diff (closest to passing):")
for d, t in outside[:8]:
    print("   ", d, t)

in-scope with the HIGHEST diff (closest to being blocked):
    0.091 أوصل للجولد إزاي؟
    0.057 ينفع أشرب شيشة في الأوضة؟
    0.055 عايز أحجز وأستخدم نقطي كخصم
    0.045 عندي حساسية من المكسرات، أم علي فيها مكسرات؟ وفي حلو تاني؟
    0.044 حاجز من بوكينج، أعدل منكم ولا منهم؟
    0.042 في كشري في المنيو؟
    0.039 عايز عربية من المطار وكمان الواي فاي بفلوس؟
    0.025 بكام السويت تلات ليالي؟

outside with the LOWEST diff (closest to passing):
    0.083 تعرف تحجزلي في مطعم برا الفندق؟
    0.09 مين أحسن، انتوا ولا الفورسيزونز؟
    0.112 الجو هيبقى عامل إيه بكرة؟
    0.125 أزور أنهي أهرامات والتذاكر بكام؟
    0.134 عايز فيزا لمصر، أعمل إيه؟
    0.142 تذكرة الأهرامات بكام؟
    0.161 احجزلي طيارة من أسوان للقاهرة
    0.168 احجزلي تذكرة طيران للأقصر


In [3]:
def check(margin):
    s.OOS_MARGIN = margin
    hits = 0
    for item in positives:
        sources = []
        for r in s.search(item["question"]):
            sources.append(r["source"])
        if any(d in sources for d in item["docs"]):
            hits += 1
    blocked = 0
    for t in refuse + faq_out:
        if not isinstance(t, str):
            t = t["question"]
        if s.search(t) == []:
            blocked += 1
    return round(100 * hits / len(positives)), blocked

for m in [0.05, 0.06, 0.07, 0.08]:
    print(m, check(m))

0.05 (88, 27)
0.06 (89, 27)
0.07 (89, 27)
0.08 (89, 27)


In [4]:
s.OOS_MARGIN = 0.07

for text in ["أوصل للجولد إزاي؟", "عايز أحجز وأستخدم نقطي كخصم", "ينفع أشرب شيشة في الأوضة؟"]:
    print(text)
    for r in s.search(text):
        print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

أوصل للجولد إزاي؟
عايز أحجز وأستخدم نقطي كخصم
    0.5233 booking_process.txt | Q: عايز أحجز، أعمل إيه؟ A: تختار التواريخ ونوع الأوضة، تدّين
    0.5188 booking_modification.txt | Q: نسيت رقم الحجز خالص، أعمل إيه؟ A: كلّم خدمة العملاء وقوله
    0.5147 room_policies.txt | Q: ينفع أختار رقم الأوضة؟ A: تخصيص الأوض على حسب المتاح وقت 
    0.5143 refund_policy.txt | Q: محتاج إيه عشان أتابع الاسترداد؟ A: ابعت بيانات حجزك لخدمة
    0.5077 loyalty_programs.txt | Q: في برنامج نقط للزباين الدايمة؟ A: أيوة، في برنامج Nile Pe
ينفع أشرب شيشة في الأوضة؟
    0.5251 complaints_resolution.txt | Q: دخلت الأوضة لقيتها مش نضيفة خالص ومش هقدر أقعد فيها A: كل
    0.5124 booking_policy.txt | Q: ينفع أحجز عن طريقكم على طول؟ A: تقدر تحجز أي أوضة متاحة، 
    0.5078 pricing_and_rates.txt | Q: الأسعار دي شاملة الفطار ولا لأ؟ A: لأ، سعر الأوضة من غير 
    0.5071 room_policies.txt | Q: ينفع أختار رقم الأوضة؟ A: تخصيص الأوض على حسب المتاح وقت 
    0.4998 pricing_and_rates.txt | Q: أرخص أوضة عندكم بكام؟ A: أرخص حاجة عن

In [5]:
import csv
from my_rag.config import FAQ_FILE

NEW = [
    ("FAQ3644", "إزاي أبقى جولد عندكم؟"),
    ("FAQ3644", "البلاتينيوم محتاج كام ليلة؟"),
    ("FAQ3644", "إيه الفرق بين السيلفر والجولد والبلاتينيوم؟"),
    ("FAQ3645", "أستخدم البوينتس بتاعتي إزاي؟"),
    ("FAQ3645", "ينفع أخصم النقط من تمن الأوضة؟"),
    ("FAQ3776", "الشيشة مسموحة جوه الفندق؟"),
    ("FAQ3776", "فين أقدر أشرب شيشة أو سجاير؟"),
]

with open(FAQ_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fields = reader.fieldnames
    rows = list(reader)

by_id = {}
questions = set()
max_id = 0
for r in rows:
    by_id[r["faq_id"]] = r
    questions.add(r["question"])
    max_id = max(max_id, int(r["faq_id"].replace("FAQ", "")))

added = []
for base_id, question in NEW:
    if question in questions:
        continue
    max_id += 1
    row = dict(by_id[base_id])
    row["faq_id"] = f"FAQ{max_id}"
    row["question"] = question
    added.append(row)

with open(FAQ_FILE, "rb") as f:
    f.seek(-1, 2)
    last = f.read(1)

with open(FAQ_FILE, "a", encoding="utf-8", newline="") as f:
    if last != b"\n":
        f.write("\r\n")
    writer = csv.DictWriter(f, fieldnames=fields)
    for row in added:
        writer.writerow(row)

print(len(rows), "->", len(rows) + len(added))

3996 -> 4003


In [6]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s

anchors = s._load_anchors()
q = s._encode("أوصل للجولد إزاي؟")
a = s._anchor_vecs @ q
c = s._vectors @ q

order = a.argsort()[::-1][:3]
for i in order:
    print("anchor", round(float(a[i]), 3), "|", anchors[i])

order = c.argsort()[::-1][:3]
for i in order:
    print("chunk ", round(float(c[i]), 3), "|", s._chunks[i]["source"], "|", s._chunks[i]["text"][:60].replace("\n", " "))

anchor 0.388 | حاسس بوجع في صدري، آخد إيه؟
anchor 0.367 | عندي ألم في صدري، آخد إيه؟
anchor 0.362 | ترشحلي فيلم أتفرج عليه؟
chunk  0.296 | contact_information.txt | Q: أكلم الحجوزات على رقم إيه؟ A: عنوان الفندق 14 شارع كورنيش
chunk  0.296 | room_policies.txt | Q: حاجز ومأكد، ووصلت قالولي مفيش أوض فاضية A: تخصيص الأوض عل
chunk  0.295 | check_in_check_out.txt | Q: إمتى أقدر آخد مفتاح الأوضة؟ A: الدخول من الساعة 3 العصر.


In [ ]:
for text in ["أوصل للجولد إزاي؟", "إزاي أوصل لمستوى Gold في برنامج الولاء؟", "loyalty program gold tier requirements"]:
    print(text)
    for r in s.search(text)[:3]:
        print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

أوصل للجولد إزاي؟
إزاي أوصل لمستوى Gold في برنامج الولاء؟
    0.5462 loyalty_programs.txt | Q: أنا نزيل حالياً في فندق Nile Pearl Hotel وأحتاج مساعدة بخ
    0.5299 loyalty_programs.txt | Q: أود الحصول على معلومات حول مستويات عضوية برنامج الولاء. A
    0.524 loyalty_programs.txt | Q: أوصل للمستوى الذهبي إزاي؟ A: برنامج Nile Pearl Rewards في
loyalty program gold tier requirements
    0.6718 loyalty_programs.txt | LOYALTY PROGRAMS - 2. Membership Tiers Standard: on joining 
    0.6437 loyalty_programs.txt | Q: Could you clarify where to get full loyalty program detai
    0.6381 loyalty_programs.txt | LOYALTY PROGRAMS - 5. Tier Benefits Silver: late check-out u


: 